# Data Processing Utilities Demo

This notebook demonstrates the usage of all utility functions in the `src/data_processing.py` module.

**Author:** Emmanuel  
**Purpose:** Test and validate data processing functions for Task C

### Loading Data Function

In [2]:
# Import required libraries
import pandas as pd
import numpy as np
import sys
from pathlib import Path

# Add src directory to path to import our module
sys.path.append('../src')

# Import our data processing utilities
# Update imports to include the new function
from data_processing import (load_and_combine_data, create_risk_categories, 
                              engineer_features, split_and_scale_data, load_processed_data)

df = load_and_combine_data()


✓ Dataset loaded successfully: 1044 student-course records
✓ Columns: 35


In [3]:
# Validate the data
print("\n" + "=" * 60)
print("Data Validation:")
print("=" * 60)
print(f"✓ Total records: {len(df)}")
print(f"✓ Total features: {df.shape[1]}")
print(f"✓ Missing values per column:\n{df.isnull().sum()}")
print(f"\n✓ Unique subjects: {df['subject'].unique()}")
print(f"✓ Existing risk categories: {df['risk_category'].unique()}")



Data Validation:
✓ Total records: 1044
✓ Total features: 35
✓ Missing values per column:
school           0
sex              0
age              0
address          0
famsize          0
Pstatus          0
Medu             0
Fedu             0
Mjob             0
Fjob             0
reason           0
guardian         0
traveltime       0
studytime        0
failures         0
schoolsup        0
famsup           0
paid             0
activities       0
nursery          0
higher           0
internet         0
romantic         0
famrel           0
freetime         0
goout            0
Dalc             0
Walc             0
health           0
absences         0
G1               0
G2               0
G3               0
subject          0
risk_category    0
dtype: int64

✓ Unique subjects: ['math' 'portuguese']
✓ Existing risk categories: ['High' 'Medium' 'Low']


### Risk Categories function

This function creates risk categories (Low, Medium, High) based on final grades (G3).

In [4]:
# Test the create_risk_categories function
print("Testing create_risk_categories()...")
print("=" * 60)

# Create a copy to preserve original data
df_with_risk = df.copy()

# Apply risk categorization
df_with_risk = create_risk_categories(df_with_risk)

print("\n" + "=" * 60)
print("Risk Category Analysis:")
print("=" * 60)
print(f"\nRisk category column added: {'risk_category' in df_with_risk.columns}")
print(f"\nSample of categorized data:")
print(df_with_risk[['G1', 'G2', 'G3', 'risk_category']].head(10))


Testing create_risk_categories()...
✓ Risk categories created successfully
✓ Distribution:
risk_category
High      230
Low       294
Medium    520
Name: count, dtype: int64

Risk Category Analysis:

Risk category column added: True

Sample of categorized data:
   G1  G2  G3 risk_category
0   5   6   6          High
1   5   5   6          High
2   7   8  10        Medium
3  15  14  15           Low
4   6  10  10        Medium
5  15  15  15           Low
6  12  12  11        Medium
7   6   5   6          High
8  16  18  19           Low
9  14  15  15           Low


In [5]:
# Validate risk categories
print("\n" + "=" * 60)
print("Category Validation:")
print("=" * 60)

# Check category distribution
print("\nCategory counts:")
print(df_with_risk['risk_category'].value_counts())

# Verify categorization logic
print("\nG3 ranges by risk category:")
print(df_with_risk.groupby('risk_category')['G3'].agg(['min', 'max', 'mean', 'count']))

# Check for any miscategorizations
print("\nVerifying categorization rules:")
high_risk = df_with_risk[df_with_risk['risk_category'] == 'High']['G3']
medium_risk = df_with_risk[df_with_risk['risk_category'] == 'Medium']['G3']
low_risk = df_with_risk[df_with_risk['risk_category'] == 'Low']['G3']

print(f"High risk (G3 < 10): All values < 10? {(high_risk < 10).all()}")
print(f"Medium risk (10 <= G3 <= 13): All values in range? {((medium_risk >= 10) & (medium_risk <= 13)).all()}")
print(f"Low risk (G3 >= 14): All values >= 14? {(low_risk >= 14).all()}")



Category Validation:

Category counts:
risk_category
Medium    520
Low       294
High      230
Name: count, dtype: int64

G3 ranges by risk category:
               min  max       mean  count
risk_category                            
High             0    9   6.039130    230
Low             14   20  15.496599    294
Medium          10   13  11.338462    520

Verifying categorization rules:
High risk (G3 < 10): All values < 10? True
Medium risk (10 <= G3 <= 13): All values in range? True
Low risk (G3 >= 14): All values >= 14? True


In [6]:
# Test error handling
print("\n" + "=" * 60)
print("Testing Error Handling:")
print("=" * 60)

# Test with empty DataFrame
try:
    empty_df = pd.DataFrame()
    create_risk_categories(empty_df)
except ValueError as e:
    print(f"✓ Empty DataFrame error caught: {e}")

# Test with missing G3 column
try:
    bad_df = pd.DataFrame({'G1': [10, 15], 'G2': [12, 14]})
    create_risk_categories(bad_df)
except KeyError as e:
    print(f"✓ Missing column error caught: Column 'G3' not found")



Testing Error Handling:
Error: Input DataFrame is empty.
✓ Empty DataFrame error caught: Input DataFrame is empty.
Error: "Column 'G3' not found in DataFrame. Available columns: G1, G2"
✓ Missing column error caught: Column 'G3' not found


### Engineer Features Function

This function creates new derived features from existing student data to improve model performance.

In [7]:
# Test the engineer_features function
print("Testing engineer_features()...")
print("=" * 60)

# Create a copy to preserve original data
df_engineered = df.copy()

# Store original shape
original_shape = df_engineered.shape

# Apply feature engineering
df_engineered = engineer_features(df_engineered)

print("\n" + "=" * 60)
print("Feature Engineering Results:")
print("=" * 60)
print(f"Original shape: {original_shape}")
print(f"New shape: {df_engineered.shape}")
print(f"Features added: {df_engineered.shape[1] - original_shape[1]}")


Testing engineer_features()...
Creating engineered features...
✓ Created 'grade_improvement' (G3 - G1)
✓ Created 'grade_trend' (average grade change per period)
✓ Created 'total_alcohol' (Dalc + Walc)
✓ Created 'study_time_binary' (studytime >= 3)
✓ Created 'failure_flag' (failures > 0)
✓ Created 'parent_education' (average of Medu and Fedu)
✓ Created 'social_score' (goout + freetime)
✓ Feature engineering completed successfully!
✓ Total new features created: 7
✓ New dataset shape: (1044, 42)

Feature Engineering Results:
Original shape: (1044, 35)
New shape: (1044, 42)
Features added: 7


In [8]:
# Validate new features
print("\n" + "=" * 60)
print("New Features Validation:")
print("=" * 60)

new_features = ['grade_improvement', 'grade_trend', 'total_alcohol', 
                'study_time_binary', 'failure_flag', 'parent_education', 'social_score']

print("\nChecking if all new features exist:")
for feature in new_features:
    exists = feature in df_engineered.columns
    print(f"  {feature}: {'✓' if exists else '✗'}")

print("\n" + "=" * 60)
print("Sample of New Features with Context:")
print("=" * 60)

# Create a more descriptive view
sample_display = df_engineered[new_features].head(10).copy()

# Rename columns for display with units/context
sample_display.columns = [
    'Grade Improvement (points)',
    'Grade Trend (avg points/period)',
    'Total Alcohol (days/week)',
    'High Study Time (1=yes, 0=no)',
    'Has Failures (1=yes, 0=no)',
    'Parent Education (0-4 scale)',
    'Social Score (combined scale)'
]

print(sample_display)



New Features Validation:

Checking if all new features exist:
  grade_improvement: ✓
  grade_trend: ✓
  total_alcohol: ✓
  study_time_binary: ✓
  failure_flag: ✓
  parent_education: ✓
  social_score: ✓

Sample of New Features with Context:
   Grade Improvement (points)  Grade Trend (avg points/period)  \
0                           1                              0.5   
1                           1                              0.5   
2                           3                              1.5   
3                           0                              0.0   
4                           4                              2.0   
5                           0                              0.0   
6                          -1                             -0.5   
7                           0                              0.0   
8                           3                              1.5   
9                           1                              0.5   

   Total Alcohol (days/week)  Hi

In [9]:
# Verify feature calculations
print("\n" + "=" * 60)
print("Feature Calculation Verification:")
print("=" * 60)

# Sample a few rows to verify calculations
sample = df_engineered[['G1', 'G2', 'G3', 'grade_improvement', 'grade_trend', 
                         'Dalc', 'Walc', 'total_alcohol', 'studytime', 'study_time_binary',
                         'failures', 'failure_flag', 'Medu', 'Fedu', 'parent_education']].head(5)

print("\nVerifying calculations on sample data:")
print(sample)

# Verify logic for specific features
print("\n" + "=" * 60)
print("Feature Statistics:")
print("=" * 60)

print(f"\nGrade Improvement (points change from G1 to G3):")
print(df_engineered['grade_improvement'].describe())

print(f"\nGrade Trend (average points change per period):")
print(df_engineered['grade_trend'].describe())

print(f"\nTotal Alcohol (combined weekday + weekend consumption, scale 2-10):")
print(f"  Min: {df_engineered['total_alcohol'].min()}, Max: {df_engineered['total_alcohol'].max()}")
print(f"  Mean: {df_engineered['total_alcohol'].mean():.2f}")

print(f"\nStudy Time Binary (1 = high study time [≥3 hours], 0 = low):")
print(df_engineered['study_time_binary'].value_counts())

print(f"\nFailure Flag (1 = has past failures, 0 = no failures):")
print(df_engineered['failure_flag'].value_counts())

print(f"\nParent Education (average education level, scale 0-4):")
print(f"  Min: {df_engineered['parent_education'].min()}, Max: {df_engineered['parent_education'].max()}")
print(f"  Mean: {df_engineered['parent_education'].mean():.2f}")

print(f"\nSocial Score (combined going out + free time, scale 2-10):")
print(f"  Min: {df_engineered['social_score'].min()}, Max: {df_engineered['social_score'].max()}")
print(f"  Mean: {df_engineered['social_score'].mean():.2f}")



Feature Calculation Verification:

Verifying calculations on sample data:
   G1  G2  G3  grade_improvement  grade_trend  Dalc  Walc  total_alcohol  \
0   5   6   6                  1          0.5     1     1              2   
1   5   5   6                  1          0.5     1     1              2   
2   7   8  10                  3          1.5     2     3              5   
3  15  14  15                  0          0.0     1     1              2   
4   6  10  10                  4          2.0     1     2              3   

   studytime  study_time_binary  failures  failure_flag  Medu  Fedu  \
0          2                  0         0             0     4     4   
1          2                  0         0             0     1     1   
2          2                  0         3             1     1     1   
3          3                  1         0             0     4     2   
4          2                  0         0             0     3     3   

   parent_education  
0               4.0

In [10]:
# Test error handling
print("\n" + "=" * 60)
print("Testing Error Handling:")
print("=" * 60)

# Test with empty DataFrame
try:
    empty_df = pd.DataFrame()
    engineer_features(empty_df)
except ValueError as e:
    print(f"✓ Empty DataFrame error caught: {e}")

# Test with missing required columns
try:
    bad_df = pd.DataFrame({'G1': [10, 15], 'G2': [12, 14]})
    engineer_features(bad_df)
except KeyError as e:
    print(f"✓ Missing columns error caught: Missing required columns detected")



Testing Error Handling:
Error: Input DataFrame is empty.
✓ Empty DataFrame error caught: Input DataFrame is empty.
Error: 'Missing required columns: G3, Dalc, Walc, studytime, failures, Medu, Fedu, goout, freetime. Available columns: G1, G2'
✓ Missing columns error caught: Missing required columns detected


### Split and Scale Data Function

This function splits data into train/test sets and scales features using StandardScaler.

In [11]:
# Test the split_and_scale_data function
print("Testing split_and_scale_data()...")
print("=" * 60)

# Prepare data for splitting
# Select only numeric columns for this demo
numeric_cols = df_engineered.select_dtypes(include=[np.number]).columns.tolist()

# Remove target variable and any ID columns
X = df_engineered[numeric_cols].drop(['G3'], axis=1, errors='ignore')
y = df_engineered['G3']

print(f"Feature matrix shape: {X.shape}")
print(f"Target variable shape: {y.shape}")
print(f"\nFeatures being used: {X.columns.tolist()}")

# Apply split and scale
X_train, X_test, y_train, y_test, scaler = split_and_scale_data(X, y)


Testing split_and_scale_data()...
Feature matrix shape: (1044, 22)
Target variable shape: (1044,)

Features being used: ['age', 'Medu', 'Fedu', 'traveltime', 'studytime', 'failures', 'famrel', 'freetime', 'goout', 'Dalc', 'Walc', 'health', 'absences', 'G1', 'G2', 'grade_improvement', 'grade_trend', 'total_alcohol', 'study_time_binary', 'failure_flag', 'parent_education', 'social_score']
Splitting and scaling data...
✓ Data split completed (80-20 split)
  Training samples: 835
  Testing samples: 209
✓ Feature scaling completed (StandardScaler)
  Features scaled: 22
  Mean of scaled training data: 0.000000
  Std of scaled training data: 1.000000
✓ Split and scale completed successfully!


In [12]:
# Validate split and scaling
print("\n" + "=" * 60)
print("Split and Scale Validation:")
print("=" * 60)

print(f"\nData Split:")
print(f"  Total samples: {len(X)}")
print(f"  Training set: {len(X_train)} ({len(X_train)/len(X)*100:.1f}%)")
print(f"  Test set: {len(X_test)} ({len(X_test)/len(X)*100:.1f}%)")

print(f"\nTarget Distribution:")
print(f"  Training y - Mean: {y_train.mean():.2f}, Std: {y_train.std():.2f}")
print(f"  Test y - Mean: {y_test.mean():.2f}, Std: {y_test.std():.2f}")

print(f"\nScaling Verification:")
print(f"  X_train scaled - Mean: {X_train.mean():.6f}, Std: {X_train.std():.6f}")
print(f"  X_test scaled - Mean: {X_test.mean():.6f}, Std: {X_test.std():.6f}")
print(f"  (Should be close to 0 mean and 1 std)")

print(f"\nScaler object type: {type(scaler)}")
print(f"Feature names from original data: {X.columns.tolist()[:5]}... (showing first 5)")



Split and Scale Validation:

Data Split:
  Total samples: 1044
  Training set: 835 (80.0%)
  Test set: 209 (20.0%)

Target Distribution:
  Training y - Mean: 11.41, Std: 3.84
  Test y - Mean: 11.06, Std: 3.94

Scaling Verification:
  X_train scaled - Mean: 0.000000, Std: 1.000000
  X_test scaled - Mean: -0.033853, Std: 1.025280
  (Should be close to 0 mean and 1 std)

Scaler object type: <class 'sklearn.preprocessing._data.StandardScaler'>
Feature names from original data: ['age', 'Medu', 'Fedu', 'traveltime', 'studytime']... (showing first 5)


In [13]:
# Test error handling
print("\n" + "=" * 60)
print("Testing Error Handling:")
print("=" * 60)

# Test with empty data
try:
    empty_X = pd.DataFrame()
    empty_y = pd.Series()
    split_and_scale_data(empty_X, empty_y)
except ValueError as e:
    print(f"✓ Empty data error caught: Feature matrix X is empty")

# Test with mismatched shapes
try:
    bad_X = pd.DataFrame({'A': [1, 2, 3]})
    bad_y = pd.Series([1, 2])
    split_and_scale_data(bad_X, bad_y)
except ValueError as e:
    print(f"✓ Shape mismatch error caught: Shape mismatch detected")



Testing Error Handling:
Error: Feature matrix X is empty.
✓ Empty data error caught: Feature matrix X is empty
Error: Shape mismatch: X has 3 samples but y has 2 samples.
✓ Shape mismatch error caught: Shape mismatch detected


### Load Processed Data Function

This function combines all preprocessing steps into one convenient pipeline.

In [14]:
# Test the load_processed_data function
print("Testing load_processed_data()...")
print("=" * 60)

# Load all processed data in one step
X_ready, y_ready, df_full = load_processed_data()


Testing load_processed_data()...
Loading and preprocessing data...

[1/3] Loading combined dataset...
✓ Dataset loaded successfully: 1044 student-course records
✓ Columns: 35

[2/3] Creating risk categories...
✓ Risk categories created successfully
✓ Distribution:
risk_category
High      230
Low       294
Medium    520
Name: count, dtype: int64

[3/3] Engineering features...
Creating engineered features...
✓ Created 'grade_improvement' (G3 - G1)
✓ Created 'grade_trend' (average grade change per period)
✓ Created 'total_alcohol' (Dalc + Walc)
✓ Created 'study_time_binary' (studytime >= 3)
✓ Created 'failure_flag' (failures > 0)
✓ Created 'parent_education' (average of Medu and Fedu)
✓ Created 'social_score' (goout + freetime)
✓ Feature engineering completed successfully!
✓ Total new features created: 7
✓ New dataset shape: (1044, 42)

Preparing feature matrix and target variable...
✓ Feature matrix created: (1044, 20)
✓ Target variable created: (1044,)
✓ Features included: 20

Feature l

In [15]:
# Validate the processed data
print("\n" + "=" * 60)
print("Processed Data Validation:")
print("=" * 60)

print(f"\nFeature Matrix (X):")
print(f"  Shape: {X_ready.shape}")
print(f"  Data type: {type(X_ready)}")
print(f"  Columns: {X_ready.columns.tolist()}")

print(f"\nTarget Variable (y):")
print(f"  Shape: {y_ready.shape}")
print(f"  Data type: {type(y_ready)}")
print(f"  Distribution:")
print(y_ready.describe())

print(f"\nFull DataFrame:")
print(f"  Shape: {df_full.shape}")
print(f"  Includes risk_category: {'risk_category' in df_full.columns}")
print(f"  Includes engineered features: {'grade_improvement' in df_full.columns}")



Processed Data Validation:

Feature Matrix (X):
  Shape: (1044, 20)
  Data type: <class 'pandas.core.frame.DataFrame'>
  Columns: ['age', 'Medu', 'Fedu', 'traveltime', 'studytime', 'failures', 'famrel', 'freetime', 'goout', 'Dalc', 'Walc', 'health', 'absences', 'grade_improvement', 'grade_trend', 'total_alcohol', 'study_time_binary', 'failure_flag', 'parent_education', 'social_score']

Target Variable (y):
  Shape: (1044,)
  Data type: <class 'pandas.core.series.Series'>
  Distribution:
count    1044.000000
mean       11.341954
std         3.864796
min         0.000000
25%        10.000000
50%        11.000000
75%        14.000000
max        20.000000
Name: G3, dtype: float64

Full DataFrame:
  Shape: (1044, 42)
  Includes risk_category: True
  Includes engineered features: True


In [16]:
# Test end-to-end pipeline
print("\n" + "=" * 60)
print("End-to-End Pipeline Test:")
print("=" * 60)

print("\nTesting complete preprocessing + split/scale pipeline...")

# Use the processed data directly with split_and_scale
X_train_final, X_test_final, y_train_final, y_test_final, scaler_final = split_and_scale_data(X_ready, y_ready)

print("\n✓ Complete pipeline successful!")
print(f"  Data loaded and preprocessed: {len(df_full)} samples")
print(f"  Features engineered: {X_ready.shape[1]} features")
print(f"  Train set: {len(X_train_final)} samples")
print(f"  Test set: {len(X_test_final)} samples")
print(f"  Data scaled and ready for modeling!")



End-to-End Pipeline Test:

Testing complete preprocessing + split/scale pipeline...
Splitting and scaling data...
✓ Data split completed (80-20 split)
  Training samples: 835
  Testing samples: 209
✓ Feature scaling completed (StandardScaler)
  Features scaled: 20
  Mean of scaled training data: 0.000000
  Std of scaled training data: 1.000000
✓ Split and scale completed successfully!

✓ Complete pipeline successful!
  Data loaded and preprocessed: 1044 samples
  Features engineered: 20 features
  Train set: 835 samples
  Test set: 209 samples
  Data scaled and ready for modeling!


In [19]:
# Summary of all functions
print("\n" + "=" * 60)
print("SUMMARY: All Data Processing Functions Tested")
print("=" * 60)

functions_tested = [
    "1. load_and_combine_data() - ✓ Loads dataset",
    "2. create_risk_categories() - ✓ Creates risk levels",
    "3. engineer_features() - ✓ Creates 7 new features",
    "4. split_and_scale_data() - ✓ Splits and scales data",
    "5. load_processed_data() - ✓ Complete preprocessing pipeline"
]

for func in functions_tested:
    print(f"  {func}")

print("\n" + "=" * 60)
print("All utility functions working correctly!")
print("=" * 60)



SUMMARY: All Data Processing Functions Tested
  1. load_and_combine_data() - ✓ Loads dataset
  2. create_risk_categories() - ✓ Creates risk levels
  3. engineer_features() - ✓ Creates 7 new features
  4. split_and_scale_data() - ✓ Splits and scales data
  5. load_processed_data() - ✓ Complete preprocessing pipeline

All utility functions working correctly!
